# nb03: Linear vs Cosine Schedule 可视化对比

**对应讲义**: L05 (Improved DDPM)  
**目标**: 直观理解为什么 cosine schedule 比 linear 更好。

本 notebook 不需要 GPU，5 分钟跑完。

## 实验内容

1. 实现 linear 和 cosine 两种 β schedule
2. 对比 $\bar\alpha_t$、$\beta_t$、SNR 随 $t$ 的变化
3. 可视化两种 schedule 下 CIFAR-10 图像的加噪过程
4. 找出关键差异时间区间

In [ ]:
import math
import torch
import numpy as np
import matplotlib.pyplot as plt

T = 1000

# --- Linear schedule ---
def linear_beta_schedule(T, beta_start=1e-4, beta_end=0.02):
    return torch.linspace(beta_start, beta_end, T)

# --- Cosine schedule (Nichol & Dhariwal 2021) ---
def cosine_beta_schedule(T, s=0.008):
    steps = T + 1
    t = torch.linspace(0, T, steps) / T
    f = torch.cos((t + s) / (1 + s) * math.pi / 2) ** 2
    alpha_bar = f / f[0]
    betas = 1 - alpha_bar[1:] / alpha_bar[:-1]
    return torch.clip(betas, 1e-5, 0.999)

betas_lin = linear_beta_schedule(T)
betas_cos = cosine_beta_schedule(T)

alphas_bar_lin = torch.cumprod(1 - betas_lin, dim=0)
alphas_bar_cos = torch.cumprod(1 - betas_cos, dim=0)

print('Linear: β_0={:.5f}, β_T={:.5f}, ᾱ_T={:.5e}'.format(
    betas_lin[0].item(), betas_lin[-1].item(), alphas_bar_lin[-1].item()))
print('Cosine: β_0={:.5f}, β_T={:.5f}, ᾱ_T={:.5e}'.format(
    betas_cos[0].item(), betas_cos[-1].item(), alphas_bar_cos[-1].item()))

## §1 三视图对比

In [ ]:
t = torch.arange(T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) β_t
axes[0].plot(t, betas_lin, label='Linear', lw=2)
axes[0].plot(t, betas_cos, label='Cosine', lw=2)
axes[0].set_xlabel('t')
axes[0].set_ylabel(r'$\beta_t$')
axes[0].set_title(r'$\beta_t$ schedule')
axes[0].legend()
axes[0].grid(alpha=0.3)

# (b) ᾱ_t
axes[1].plot(t, alphas_bar_lin, label='Linear', lw=2)
axes[1].plot(t, alphas_bar_cos, label='Cosine', lw=2)
axes[1].set_xlabel('t')
axes[1].set_ylabel(r'$\bar\alpha_t$')
axes[1].set_title('Signal preservation')
axes[1].legend()
axes[1].grid(alpha=0.3)

# (c) log SNR
snr_lin = alphas_bar_lin / (1 - alphas_bar_lin)
snr_cos = alphas_bar_cos / (1 - alphas_bar_cos)
axes[2].plot(t, torch.log(snr_lin), label='Linear', lw=2)
axes[2].plot(t, torch.log(snr_cos), label='Cosine', lw=2)
axes[2].set_xlabel('t')
axes[2].set_ylabel('log SNR')
axes[2].set_title('Signal-to-Noise Ratio')
axes[2].legend()
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('schedule_compare.png', dpi=100, bbox_inches='tight')
plt.show()

**观察**:
- 在 (b) 中，cosine 的 $\bar\alpha_t$ 在中间区域下降更平缓
- 在 (c) 中，cosine 在 $t \in [200, 800]$ 的 SNR 范围更宽，意味着这个区间有更多"有意义的加噪状态"供学习

## §2 关键时间点对比

In [ ]:
key_ts = [0, 100, 250, 500, 750, 900, 999]
print(f"{'t':>5} | {'ᾱ_lin':>10} | {'ᾱ_cos':>10} | {'σ_lin':>8} | {'σ_cos':>8}")
print('-' * 60)
for ti in key_ts:
    al, ac = alphas_bar_lin[ti].item(), alphas_bar_cos[ti].item()
    sl, sc = math.sqrt(1 - al), math.sqrt(1 - ac)
    print(f"{ti:>5} | {al:>10.4f} | {ac:>10.4f} | {sl:>8.4f} | {sc:>8.4f}")

**关键差异**:
- 在 t=250 时，linear 已经噪声占主导 (σ≈0.6)，cosine 仍保留较多信号 (σ≈0.4)
- 这就是 "cosine 给中间时间步更多有效训练机会" 的具体含义

## §3 在 CIFAR-10 图像上可视化加噪过程

In [ ]:
from torchvision import datasets, transforms

# 加载一张 CIFAR-10 图
tx = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
ds = datasets.CIFAR10(root='./data', train=True, download=True, transform=tx)
x0, _ = ds[7]  # 任一张图
x0 = x0.unsqueeze(0)  # (1, 3, 32, 32)

ts_show = [0, 100, 250, 500, 750, 900, 999]
fig, axes = plt.subplots(2, len(ts_show), figsize=(2*len(ts_show), 4))

torch.manual_seed(0)
noise = torch.randn_like(x0)

for col, ti in enumerate(ts_show):
    # Linear
    a_lin = alphas_bar_lin[ti]
    x_lin = a_lin.sqrt() * x0 + (1 - a_lin).sqrt() * noise
    axes[0, col].imshow(((x_lin[0].permute(1,2,0) + 1) / 2).clamp(0, 1))
    axes[0, col].set_title(f't={ti}')
    axes[0, col].axis('off')
    
    # Cosine
    a_cos = alphas_bar_cos[ti]
    x_cos = a_cos.sqrt() * x0 + (1 - a_cos).sqrt() * noise
    axes[1, col].imshow(((x_cos[0].permute(1,2,0) + 1) / 2).clamp(0, 1))
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Linear', fontsize=12)
axes[1, 0].set_ylabel('Cosine', fontsize=12)
plt.tight_layout()
plt.savefig('noising_compare.png', dpi=100, bbox_inches='tight')
plt.show()

**直观结论**: cosine schedule 下，图像内容在 t=500 时仍可辨认；linear 下早已无法辨认。

这就是为什么 cosine 训练出来的模型 NLL 更好——它在中间区间有更多有信号的样本来学习。

## §4 实验任务

1. 用 32×32 vs 64×64 vs 256×256 不同分辨率的图重做 §3 实验，体会 cosine 在不同尺度下的差异
2. 实现 `sigmoid` schedule（更陡峭的版本）并加入对比
3. 计算两种 schedule 下，使用 simplified loss 时每个 t 的等效"训练权重"差异

## 参考
- Nichol & Dhariwal, *Improved DDPM*, ICML 2021
- L05 讲义 §2